# Pipeline Walkthrough (Production Modules Only)

This notebook replaces the old 01/03/04/05 exploration notebooks (now archived under notebooks/legacy/).
Instead of demoing standalone rule-based helper functions, this notebook calls the real production
src/ modules in the same order src/pipeline.py does, so it always reflects what actually runs.

Pipeline stage order: tokenizer -> bert_ner -> entity_merger -> classifier -> report_generator


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

## 1. Load a raw sample report

Dataset used: data/raw/sample_reports/testing/report_001.txt (synthetic sample text).

In [ ]:
REPORT_PATH = PROJECT_ROOT / "data/raw/sample_reports/testing/report_001.txt"
raw_text = REPORT_PATH.read_text(encoding="utf-8")
print(raw_text)

## 2. Clean / preprocess the text

Module: src.tokenizer.preprocess_report (text cleaning only, no dataset dependency).

In [ ]:
from src.tokenizer import preprocess_report

cleaned_text = preprocess_report(raw_text)
print(cleaned_text)

## 3. Extract entities with the fine-tuned BERT NER model

Module: src.bert_ner.TransformerNER. Model is fine-tuned on the AnnoCTR dataset (data/external/annoctr/) if a local checkpoint exists in config.NER_MODEL_DIR, otherwise it falls back to the pretrained hub model.

In [ ]:
from src.bert_ner import TransformerNER

ner_model = TransformerNER()
raw_entities = ner_model.extract(cleaned_text)
raw_entities

## 4. Merge / deduplicate entities

Module: src.entity_merger.build_entity_report (pure post-processing of model predictions, no dictionary lookups).

In [ ]:
from src.entity_merger import build_entity_report

entity_report = build_entity_report(raw_entities)
entity_report

## 5. Predict severity

Module: src.classifier.classify_severity. Model is a Random Forest trained in train_classifier.py on real CVSS severity labels pulled from the NVD CVE dataset (data/external/nvd/), using TF-IDF text features (src.feature_engineering).

In [ ]:
from src.classifier import classify_severity

severity_result = classify_severity(cleaned_text, entity_report["merged_entities"])
severity_result

## 6. Assemble the final report

Module: src.report_generator.build_report.

In [ ]:
from src.report_generator import build_report

final_report = build_report(
    source_path=str(REPORT_PATH),
    report_text=raw_text,
    cleaned_text=cleaned_text,
    entities=entity_report,
    classification=severity_result,
)
final_report